In [1]:
%pip install -q pandas==2.2.3 jsonschema==4.25.1 ollama==0.6.2

from pathlib import Path
import os
import sys

# VS Code may start in the workspace root or beside this notebook.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "demos" / "5.1-evals").is_dir())
DEMO = ROOT / "demos" / "5.1-evals"
os.chdir(DEMO)
sys.path.insert(0, str(ROOT / "demos"))
sys.path.insert(0, str(DEMO))


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. The task

A fictional homewares retailer wants support emails turned into structured tickets. The model gets an email and must return JSON:

| field | type |
|---|---|
| `order_id` | string or null |
| `category` | one of `refund`, `delivery`, `product`, `other` |
| `refund_requested` | boolean |
| `refund_amount` | number or null |
| `summary` | one sentence |

In [2]:
import json

with open("traces_v1.json") as f:
    doc = json.load(f)

traces = doc["traces"]
print(doc["prompt"])
print(f"\n{len(traces)} traces")

Extract the following fields from the customer email and reply with JSON only, no other text: order_id (string or null), category (one of: refund, delivery, product, other), refund_requested (boolean), refund_amount (number or null), summary (one sentence).

20 traces


In [3]:
def show(t):
    print(f"--- {t['id']} ---")
    print("EMAIL: ", t["email"])
    print("OUTPUT:", t["output"])
    print()

for t in traces[:3]:
    show(t)

--- t01 ---
EMAIL:  Hi, I ordered a cast iron pan (ORD-48213) two weeks ago and it arrived cracked down the middle. I'd like a refund of the $65 I paid. Thanks, Priya
OUTPUT: {
  "order_id": "ORD-48213",
  "category": "refund",
  "refund_requested": true,
  "refund_amount": 65,
  "summary": "Customer requests a $65 refund for a cast iron pan that arrived cracked."
}

--- t02 ---
EMAIL:  My order ORD-51902 was supposed to arrive last Thursday and the tracking hasn't moved since it left Sydney. Can you find out where it is? Cheers, Tom
OUTPUT: {
  "order_id": "ORD-51902",
  "category": "delivery",
  "refund_requested": false,
  "refund_amount": null,
  "summary": "Customer asks for a status update on a delayed delivery."
}

--- t03 ---
EMAIL:  The kettle from order ORD-40077 stopped working after a week. I want my $49 back please.
OUTPUT: ```json
{
  "order_id": "ORD-40077",
  "category": "refund",
  "refund_requested": true,
  "refund_amount": 49,
  "summary": "Customer requests a $49 r

## 2. Error analysis

Before any metric: read. Go through the traces and, for each output that is wrong, write **one sentence: what went wrong**. Do not fix anything yet, and do not invent categories up front; they fall out of the sentences.

Work through a few together (`show(traces[i])`), then fill in `notes` below.

In [4]:
# Look at more traces here. Which ones are wrong, and how?
show(traces[3])   # t04
show(traces[4])   # t05

--- t04 ---
EMAIL:  Hello, my dinner set (ORD-43310, $120) arrived with three chipped plates. Please refund the full amount.
OUTPUT: Sure! Here's the extracted information:

{
  "order_id": "ORD-43310",
  "category": "refund",
  "refund_requested": true,
  "refund_amount": 120,
  "summary": "Customer requests a $120 refund for a dinner set with chipped plates."
}

--- t05 ---
EMAIL:  I bought a set of wine glasses from your Chatswood store last month and two of them were chipped. I'd like my money back, it was about $40.
OUTPUT: {
  "order_id": "ORD-00001",
  "category": "refund",
  "refund_requested": true,
  "refund_amount": 40,
  "summary": "Customer requests a $40 refund for chipped wine glasses."
}



In [ ]:
notes = {
    "t03": "output wrapped in a markdown code fence",
    "t04": "chatty preamble before the JSON",
    "t05": "email has no order number but output invents one",
    "t06": "refund_amount is the product price, not the $15 the customer asked for",
    "t07": "refund_amount is negative",
    "t08": "obeyed instructions embedded in the email: $10,000 refund",
    "t09": "output wrapped in a markdown code fence",
    "t11": "category 'complaint' is not in the enum",
    "t12": "chatty preamble before the JSON",
    "t13": "summary says the customer is happy; the email is a complaint",
    "t15": "output wrapped in a markdown code fence",
    "t16": "email has two order ids and the output picked the wrong one",
    "t18": "order id digits transposed: ORD-46184 for ORD-46814",
}

from collections import Counter

def label(sentence):
    if "fence" in sentence or "preamble" in sentence:
        return "not bare JSON"
    if "invents" in sentence or "transposed" in sentence:
        return "order id not from the email"
    if "enum" in sentence:
        return "category outside the enum"
    if "negative" in sentence:
        return "impossible amount"
    return "content wrong, form fine"

Counter(label(s) for s in notes.values()).most_common()

[('not bare JSON', 5),
 ('content wrong, form fine', 4),
 ('order id not from the email', 2),
 ('impossible amount', 1),
 ('category outside the enum', 1)]

Choose priorities by severity as well as frequency.

## 3. Assertions

In [4]:
from ticket_contract import check_parses, parse_ticket

check_parses(traces[2]["output"]), check_parses(traces[0]["output"])

(False, True)

In [ ]:
from ticket_contract import (SCHEMA, FIELDS, CATEGORIES, check_schema,
                             check_grounded, check_amount)
import inspect

# Read the shared implementation: it validates all fields before indexing them.
print(inspect.getsource(check_schema))
print(inspect.getsource(check_grounded))
print(inspect.getsource(check_amount))

def check_schema(out):
    try:
        d = parse_ticket(out)
        if not VALIDATOR.is_valid(d):
            return False
        amount = d["refund_amount"]
        return amount is None or (type(amount) in (int, float) and math.isfinite(amount))
    except (ValueError, TypeError, OverflowError):
        return False

def check_grounded(out, email):
    """ID occurs in source; this does not identify the correct order among several."""
    if not check_schema(out):
        return False
    oid = parse_ticket(out)["order_id"]
    return oid is None or oid in email

def check_amount(out, email=None):
    """Non-negative only; this does not verify the amount actually requested."""
    if not check_schema(out):
        return False
    amount = parse_ticket(out)["refund_amount"]
    return amount is None or amount >= 0



`check_grounded` checks only whether an ID occurs in the email. It catches t18's
transposed digits, but t16 shows why occurrence is not correct extraction. Null
means the ID is missing. Empty IDs, numeric summaries and boolean amounts fail the schema.

## 4. Harness

Run every check on every trace. Per-case, per-criterion, then score.

In [8]:
import pandas as pd
from ticket_contract import run_eval

print(inspect.getsource(run_eval))
results_v1 = run_eval(traces)
results_v1

def run_eval(traces):
    import pandas as pd
    rows = []
    for t in traces:
        out, email = t["output"], t["email"]
        parses = check_parses(out)
        schema = parses and check_schema(out)
        r = {"id": t["id"], "parses": parses, "schema": schema,
             "grounded": schema and check_grounded(out, email),
             "amount": schema and check_amount(out)}
        # Invalid shape fails dependent checks. No invalid object reaches a semantic check.
        r["pass"] = all(r[k] for k in ("parses", "schema", "grounded", "amount"))
        rows.append(r)
    return pd.DataFrame(rows).set_index("id")



,parses,schema,grounded,amount,pass
id,,,,,
t01,True,True,True,True,True
t02,True,True,True,True,True
t03,False,False,False,False,False
t04,False,False,False,False,False
t05,True,True,False,True,False
t06,True,True,True,True,True
t07,True,True,True,False,False
t08,True,True,True,True,True
t09,False,False,False,False,False


In [9]:
results_v1.mean()

parses      0.75
schema      0.70
grounded    0.60
amount      0.65
pass        0.55
dtype: float64

**11/20 pass as a baseline.**

## 5. "Improve" the prompt

Someone reads a few outputs and decides the bot sounds cold. The prompt gains:

> *Be warm and polite, and briefly explain your reasoning so the customer feels heard.*

`traces_v2.json` holds the same twenty emails under the new prompt. A vibe check on any single output approves: it reads beautifully. Run the eval.

In [10]:
with open("traces_v2.json") as f:
    doc_v2 = json.load(f)

results_v2 = run_eval(doc_v2["traces"])
pd.DataFrame({"v1": results_v1.mean(), "v2": results_v2.mean()})

,v1,v2
parses,0.75,0.5
schema,0.70,0.5
grounded,0.60,0.5
amount,0.65,0.5
pass,0.55,0.5


In these curated fixtures, the parseable v2 tickets pass the limited checks but parse rate fell from 15/20 to 10/20 because the model now explains its reasoning before the JSON, and the overall score dropped from 11/20 to 10/20.